# 02 — Visualize & Validate Landmarks

Visual QA for extracted landmarks. No output files — just inspection.

## What this does
- Overlays skeleton on original video frames (side-by-side)
- Shows landmark confidence heatmap over time
- Plots joint angle trajectories to verify tracking quality

## Inputs
```
data/extracted/*.npz   — landmarks (N, 33, 3), visibility (N, 33)
data/extracted/*.json  — metadata with video_path
```

## What to look for
- Skeleton should track the body cleanly in every frame
- Confidence heatmap should be mostly green (> 0.6)
- Angle trajectories should show smooth, cyclic patterns (no wild jumps)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import json
from ipywidgets import interact, IntSlider

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
# MediaPipe 33-landmark connections (body only, skip face details)
CONNECTIONS = [
    (11, 12), (11, 23), (12, 24), (23, 24),  # Torso
    (11, 13), (13, 15),  # Left arm
    (12, 14), (14, 16),  # Right arm
    (23, 25), (25, 27),  # Left leg
    (24, 26), (26, 28),  # Right leg
    (15, 17), (15, 19), (15, 21),  # Left hand
    (16, 18), (16, 20), (16, 22),  # Right hand
    (27, 29), (27, 31),  # Left foot
    (28, 30), (28, 32),  # Right foot
]

def draw_skeleton(frame, landmarks, visibility, threshold=0.5):
    """Draw skeleton overlay on a video frame."""
    h, w = frame.shape[:2]
    overlay = frame.copy()
    
    # Draw bones
    for a, b in CONNECTIONS:
        if visibility[a] >= threshold and visibility[b] >= threshold:
            pt1 = (int(landmarks[a, 0] * w), int(landmarks[a, 1] * h))
            pt2 = (int(landmarks[b, 0] * w), int(landmarks[b, 1] * h))
            cv2.line(overlay, pt1, pt2, (91, 124, 250), 2, cv2.LINE_AA)
    
    # Draw joints (skip face: indices 0-10)
    for i in range(11, 33):
        if visibility[i] >= threshold:
            pt = (int(landmarks[i, 0] * w), int(landmarks[i, 1] * h))
            color = (78, 205, 196) if visibility[i] >= 0.7 else (245, 166, 35)
            cv2.circle(overlay, pt, 4, color, -1, cv2.LINE_AA)
    
    return cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)

## Select a video to inspect

In [ ]:
extracted_dir = Path('../data/extracted')
npz_files = sorted(extracted_dir.glob('*.npz'))

for i, f in enumerate(npz_files):
    print(f'[{i}] {f.name}')

# Change this index to inspect a different video
VIDEO_IDX = 1

In [ ]:
if npz_files:
    npz_file = npz_files[VIDEO_IDX]
    data = np.load(npz_file)
    landmarks = data['landmarks']   # (N, 33, 3)
    visibility = data['visibility'] # (N, 33)
    
    json_file = npz_file.with_suffix('.json')
    with open(json_file) as f:
        meta = json.load(f)
    
    video_path = meta['video_path']
    n_frames = len(landmarks)
    print(f'Video: {Path(video_path).name}')
    print(f'Exercise: {meta["exercise"]}')
    print(f'Frames: {n_frames}')
    print(f'Landmarks: {landmarks.shape}')
    print(f'Usable: {meta["usable_frames"]}/{meta["extracted_frames"]} ({meta["usable_pct"]}%)')
    print(f'Avg visibility: {meta["avg_visibility"]}')
else:
    print('No extracted data found. Run notebook 01 first.')

## Side-by-side: Original vs Skeleton overlay

Use the slider to scrub through frames. Verify the skeleton tracks the body correctly.

In [ ]:
def show_frame(frame_idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print('Could not read frame')
        return
    
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    overlay = draw_skeleton(frame_rgb, landmarks[frame_idx], visibility[frame_idx])
    avg_vis = visibility[frame_idx].mean()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    ax1.imshow(frame_rgb)
    ax1.set_title(f'Frame {frame_idx} — Original')
    ax1.axis('off')
    
    ax2.imshow(overlay)
    ax2.set_title(f'Frame {frame_idx} — Skeleton (vis={avg_vis:.2f})')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

if npz_files:
    interact(show_frame, frame_idx=IntSlider(min=0, max=n_frames-1, step=1, value=0))

## Landmark confidence heatmap

Shows visibility confidence for each body joint across all frames.
Green = high confidence, red = low.

In [ ]:
if npz_files:
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Show body landmarks (11-32)
    body_vis = visibility[:, 11:].T
    body_names = [
        'L_Shoulder', 'R_Shoulder', 'L_Elbow', 'R_Elbow',
        'L_Wrist', 'R_Wrist', 'L_Pinky', 'R_Pinky',
        'L_Index', 'R_Index', 'L_Thumb', 'R_Thumb',
        'L_Hip', 'R_Hip', 'L_Knee', 'R_Knee',
        'L_Ankle', 'R_Ankle', 'L_Heel', 'R_Heel',
        'L_Foot', 'R_Foot',
    ]
    
    im = ax.imshow(body_vis, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_yticks(range(len(body_names)))
    ax.set_yticklabels(body_names, fontsize=8)
    ax.set_xlabel('Frame')
    ax.set_title('Landmark Confidence Over Time (green = good)')
    plt.colorbar(im, ax=ax, label='Visibility')
    plt.tight_layout()
    plt.show()

## Joint angle trajectories

Key angles over time — should show smooth, cyclic patterns for push-ups/lunges.

In [ ]:
from src.phase_labeler import compute_angle

if npz_files:
    l_elbow = [compute_angle(landmarks[i], 11, 13, 15) for i in range(n_frames)]
    r_elbow = [compute_angle(landmarks[i], 12, 14, 16) for i in range(n_frames)]
    l_knee = [compute_angle(landmarks[i], 23, 25, 27) for i in range(n_frames)]
    r_knee = [compute_angle(landmarks[i], 24, 26, 28) for i in range(n_frames)]
    hip_line = [compute_angle(landmarks[i], 11, 23, 27) for i in range(n_frames)]
    
    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
    
    axes[0].plot(l_elbow, label='L Elbow', alpha=0.8)
    axes[0].plot(r_elbow, label='R Elbow', alpha=0.8)
    axes[0].set_ylabel('Angle (deg)')
    axes[0].set_title('Elbow Angles')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(l_knee, label='L Knee', alpha=0.8)
    axes[1].plot(r_knee, label='R Knee', alpha=0.8)
    axes[1].set_ylabel('Angle (deg)')
    axes[1].set_title('Knee Angles')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(hip_line, label='Hip (shoulder-hip-ankle)', alpha=0.8, color='purple')
    axes[2].set_ylabel('Angle (deg)')
    axes[2].set_xlabel('Frame')
    axes[2].set_title('Hip Angle (Body Line)')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
**Next:** If extraction looks good, run `03_label_phases.ipynb` to auto-detect reps and label phases.